In [1]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import sys
from pathlib import Path
import xarray as xr
import cfgrib
import cartopy.crs as ccrs  # Projeções de mapas.
import cartopy.feature as cfeature  # Elementos geográficos.
from matplotlib.tri import Triangulation

#Import local libraries
import aux

In [ ]:
init='2025030700'
year='2025'
month='03'
base_dir='/p/projetos/monan_atm/madeleine.gacita/global_data/'
exps_data={
    "teste_autoconv5": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/REGNOL2/teste_autoconv5/"},
    "teste_congestus": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/REGNOL2/teste_congestus/"},
    "CTRL": {
        "dir": f"/p/projetos/monan_atm/madeleine.gacita/scripts_CD-CT/dataout/REGNOL2/CTRL/"},
}
extents={"Amaz_reg":{
            "extent":[-90, -30, -20, 20],
            "label":"South America"},
        # "Global":{
        #     "extent":[-180, 180, -90, 90],
        #     "label":"Global"},
        }


In [3]:
var_dict={
    "ISR": {
        "era5_name" : "ssrd",
        "era5_longname" :"surface_solar_radiation_downwards",
        "ceres_name" : "init_all_sfc_sw_dn", 
        "monan_name" : "swdnb",
        "unit" : "W m^{-2}",
        "label" : "Surface shortwave radiation downwards",
        "convert_to_flux": "yes",
        "vmin" : 0,
        "vmax" : 1300,
        "vmin_diff" : -200,
        "vmax_diff" : 200,
    },
    "ISRC": {
        "era5_name" : "ssrdc",
        "era5_longname" :"surface_solar_radiation_downward_clear_sky",
        "ceres_name" : "init_clr_sfc_sw_dn", 
        "monan_name" : "swdnbc",
        "unit" : "W m^{-2}",
        "label" : "Surface shortwave radiation downwards (clear sky)",
        "convert_to_flux": "yes",
        "vmin" : 0,
        "vmax" : 1300,
        "vmin_diff" : -200,
        "vmax_diff" : 200,
    },
    "OLR": {
        "era5_name": "ttr",
        "era5_longname" :"top_net_thermal_radiation",
        "ceres_name" : "init_all_toa_lw_up", 
        "monan_name" : "lwupt",
        "unit" : "W m^{-2}",
        "label" : "TOA Outgoing longwave radiation",
        "convert_to_flux": "yes",
        "vmin" : -400,
        "vmax" : 0
    },
    "OLRC": {
        "era5_name" : "ttrc",
        "era5_longname" : "top_net_thermal_radiation_clear_sky",
        "ceres_name" : "init_clr_toa_lw_up", 
        "monan_name" : "lwuptc",
        "unit" : "W m^{-2}",
        "label" : "TOA Outgoing longwave radiation (clear sky)",
        "convert_to_flux": "yes",
        "vmin" : -400,
        "vmax" : 0,
        "vmin_diff" : -50,
        "vmax_diff" : 50
    },
    "TISR": {
        "era5_name" : "tisr",
        "era5_longname" : "toa_incident_solar_radiation",
        "ceres_name" : "toa_sw_insol", 
        "monan_name" : "swdnt",
        "unit" : "W m^{-2}",
        "label" : "TOA incident short-wave (solar) radiation",
        "convert_to_flux": "yes",
        "vmin" : 0,
        "vmax" : 1300,
        "vmin_diff" : -50,
        "vmax_diff" : 50
    },
    "PC": {
        "era5_name" : "tclw",
        "era5_longname" :"total_column_cloud_liquid_water",
        "ceres_name" : "obs_cld_lwp", 
        "monan_name" : "precipcloud",
        "unit" : "kg*m^{-2}",
        "label" : "Total column cloud liquid water",
        "vmin" : 0.1,
        "vmax" : 1,
        "vmin_diff" : -2,
        "vmax_diff" : 2,
        "convert_to_flux": "no",
    },
    "PI": {
        "era5_name": "tciw",
        "era5_longname" :"total_column_cloud_ice_water",
        "ceres_name" : "obs_cld_iwp", 
        "monan_name" : "precipice",
        "unit" : "kg*m^{-2}",
        "label" : "Total column cloud ice water",
        "vmin" : 0.02,
        "vmax" : 0.5,
        "vmin_diff" : -0.5,
        "vmax_diff" : 0.5,
        "convert_to_flux": "no",
    },
     "RAINC": {
        "era5_name" : "cp",
        "era5_longname" : "convective_precipitation",
        "ceres_name" : "", 
        "monan_name" : "rainc",
        "unit" : "mm",
        "label" : "Convective precipitation",
        "vmin" : 1,
        "vmax" : 200,
        "vmin_diff" : -100,
        "vmax_diff" : 100,
        "convert_to_flux": "no",
    }, 
     "RAIN": {
        "era5_name" : "tp",
        "era5_longname" : "gridbox_precipitation",
        "ceres_name" : "", 
        "monan_name" : "rainnc",
        "unit" : "mm",
        "label" : "Total precipitation",
        "vmin" : 1,
        "vmax" : 200,
        "vmin_diff" : -100,
        "vmax_diff" : 100,
        "convert_to_flux": "no",
    },  
    "CAPE": {
        "era5_name" : "cape",
        "era5_longname" : "convective_available_potential_energy",
        "ceres_name" : "cape", 
        "monan_name" : "cape",
        "unit" : "J kg^{-1}",
        "label" : "Convective available potential energy",
        "convert_to_flux": "no",
        "vmin" : 0,
        "vmax" : 2000
    },
    "CIN": {
        "era5_name" : "cin",
        "era5_longname" : "convective_inhibition",
        "ceres_name" : "cin", 
        "monan_name" : "cin",
        "unit" : "J kg^{-1}",
        "label" : "Convective inhibition",
        "convert_to_flux": "no",
        "vmin" : 0,
        "vmax" : 300
    }
}

surf_flux_dict={
    "HF": {
        "era5_name" : "sshf",
        "era5_longname" : "surface_sensible_heat_flux",
        "monan_name" : "hfx", 
        "unit" : "W/m**2",
        "label" : "Sensible heat flux"
    },
    "LF": {
        "era5_name": "sslf",
        "era5_longname":"surface_latent_heat_flux",
        "monan_name": "lf", 
        "unit": "W m^{-2}",
        "label": "Latent heat flux"
    }
}

profile_vars_dict={
    "ISR": {
        "monan_name" : "ssrd",
        "ceres_name" : "adj_all_sw_dn",
        "unit" : "W m^{-2}",
        "label" : "Incoming shortwave radiation"
    },
    "OSR": {
        "monan_name" : "ssrd",
        "ceres_name" : "adj_all_sw_up", 
        "unit" : "W m^{-2}",
        "label" : "Outgoing shortwave radiation"
    },
    "ILR": {
        "monan_name" : "ttrc",
        "ceres_name" : "adj_all_lw_dn", 
        "unit" : "W m^{-2}",
        "label" : "Incoming longwave radiation"
    },
    "OLR": {
        "monan_name" : "ttr",
        "ceres_name" : "adj_all_lw_up", 
        "unit" : "W m^{-2}",
        "label" : "Outgoing longwave radiation"
    }    
}

In [4]:
def apply_lon_lat_conventions(ds):
    # Renames
    if "lon" in ds.dims:
        ds = ds.rename({"lon": "longitude"})
    if "lat" in ds.dims:
        ds = ds.rename({"lat": "latitude"})
 
    # Flip latitudes (ensure they are monotonic increasing)
    if "latitude" in ds.dims:
        lats = ds["latitude"]
        if len(lats) > 1 and lats[0] > lats[-1]:
            ds = ds.reindex(latitude=ds.latitude[::-1])
 
    # Convert longitude to [-180, 180[
    if "longitude" in ds.dims and ds["longitude"].max() > 180:
        lons = ds["longitude"]
        lons_attrs = lons.attrs
        new_lons = np.concatenate([lons[lons >= 180], lons[lons < 180]])
        ds = ds.reindex(longitude=new_lons)
        ds = ds.assign_coords(longitude=(((ds["longitude"] + 180) % 360) - 180))
        ds["longitude"].attrs = lons_attrs
    return ds

# Opening CERES SYN_1deg Ed A

In [5]:
# syn_file_arg = base_dir+"CER_SYN1deg-MHour/Terra-Aqua-MODIS_Edition4A/CER_SYN1deg-MHour_Terra-Aqua-MODIS_Edition4A_407406."+year+month+".hdf"
# path = Path(syn_file_arg)
# print(syn_file_arg)

# if not path.exists():
#     print(f"File does not exist: {path!s}")
# else:
#     from pyhdf.SD import SD, SDC
#     sd = SD(str(path), SDC.READ)
#     print("pyhdf.SD opened file — datasets:")
#     ceres_names = list(sd.datasets().keys())

# Selecting and plotting var

### Plots general settings

In [6]:

from matplotlib.colors import ListedColormap, BoundaryNorm # Lista de Cores 
cores_legenda_rgb = [
(255, 255, 255),  # Branco
(220, 220, 220),  # Cinza Claro
(180, 180, 180),  # Cinza
(20, 0, 150),     # Azul Marinho/Roxo
(0, 0, 255),      # Azul
(0, 100, 100),    # Verde Escuro/Azul Petr�leo
(0, 200, 0),      # Verde
(150, 255, 0),    # Verde Lim�o/Ciano
(255, 255, 0),    # Amarelo Claro
(255, 220, 0),    # Amarelo Escuro/Ouro
(255, 130, 0),    # Laranja
(230, 25, 25),    # Vermelho Claro
(100, 0, 0),      # Vermelho Escuro/Borgonha
]

cores_normalizadas_matplotlib = []
for r, g, b in cores_legenda_rgb:
    cores_normalizadas_matplotlib.append((r / 255.0, g / 255.0, b / 255.0))

# Criacao da colormap e os niveis (clevs)
cmap = ListedColormap(cores_normalizadas_matplotlib)
clevs = [0, 1, 2, 4, 6, 10, 15, 25, 35, 50, 75, 100, 150]
clevs_full = clevs + [40]
norm = BoundaryNorm(clevs, ncolors=len(clevs), extend='max')

In [7]:

# if var_dict[var]["convert_to_flux"]=="yes":
#     era_var_name = f'{var_dict[var]["era5_name"]}_flux'
# else:
#     era_var_name = var_dict[var]["era5_name"]
fig_path="/p/projetos/monan_atm/madeleine.gacita/figuras/"

target_lon = -60.0

In [8]:
# Open and plot MSWEP daily total precipitation
# MSWEP V3 daily files are one per calendar day, named YYYY<jday:03d>.nc
MSWEP_base = '/pesq/dados/monan/users/andre.lyra/MSWEP'
MSWEP_version = 'MSWEP_V315'
MSWEP_dir = f'{MSWEP_base}/{MSWEP_version}_{year}{month}/daily/'

init_dt = pd.to_datetime(init, format='%Y%m%d%H')
n_days = n_forecast_days if 'n_forecast_days' in dir() else 5

for lead in range(n_days):
    date = init_dt + pd.Timedelta(hours=lead * 24)
    jday = date.dayofyear
    mswep_file = f'{MSWEP_dir}/{date.year}{jday:03d}.nc'

    print(f"Lead {lead} ({date.strftime('%Y-%m-%d')}): {mswep_file}")

    if not os.path.exists(mswep_file):
        print(f"  ✗ File not found")
        continue

    ds_mswep = xr.open_dataset(mswep_file, engine="netcdf4")
    print(f"  ✓ Opened. Variables: {list(ds_mswep.data_vars.keys())}")
    print(f"  Dimensions: {dict(ds_mswep.dims)}")

    # Detect precipitation variable name
    precip_var = None
    for vname in ['precipitation', 'precip', 'tp', 'rain', 'prcp']:
        if vname in ds_mswep.data_vars:
            precip_var = vname
            break

    if precip_var is None:
        print(f"  ✗ No precipitation variable found. Available: {list(ds_mswep.data_vars.keys())}")
        ds_mswep.close()
        continue

    ds_mswep = apply_lon_lat_conventions(ds_mswep)

    for myext in extents.keys():
        extent_label = extents[myext]["label"]
        fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

        data_to_plot = ds_mswep[precip_var].squeeze()
        data_masked = data_to_plot.where(data_to_plot >= 1)

        data_masked.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            norm=norm,
            cbar_kwargs={
                'shrink': 0.5,
                'aspect': 25,
                'pad': 0.05,
                'label': f"Total precipitation (mm)",
                'extend': 'max',
            },
        )

        ax.set_extent(extents[myext]["extent"], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3)
        ax.add_feature(cfeature.STATES, linewidth=0.2)

        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 10}
        gl.ylabel_style = {'size': 10}

        ax.set_title(
            f"MSWEP Total Precipitation\n{date.strftime('%Y-%m-%d')}, {extent_label}",
            fontsize=12,
            pad=20,
        )

        plt.tight_layout()
        out_png = f"{fig_path}/MSWEP_rain_total_{date.strftime('%Y%m%d')}_{myext}.png"
        plt.savefig(out_png, dpi=150, bbox_inches='tight')
        print(f"  Saved: {out_png}")
        plt.close()

    ds_mswep.close()


Lead 0 (2025-12-01): /pesq/dados/monan/users/andre.lyra/MSWEP/MSWEP_V315_202512/daily//2025335.nc
  ✓ Opened. Variables: ['precipitation']
  Dimensions: {'time': 1, 'lat': 1800, 'lon': 3600}


/tmp/ipykernel_2546069/3220491250.py:23: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds_mswep.dims)}")


  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251201_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251201_Global.png
Lead 1 (2025-12-02): /pesq/dados/monan/users/andre.lyra/MSWEP/MSWEP_V315_202512/daily//2025336.nc
  ✓ Opened. Variables: ['precipitation']
  Dimensions: {'time': 1, 'lat': 1800, 'lon': 3600}


/tmp/ipykernel_2546069/3220491250.py:23: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds_mswep.dims)}")


  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251202_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251202_Global.png
Lead 2 (2025-12-03): /pesq/dados/monan/users/andre.lyra/MSWEP/MSWEP_V315_202512/daily//2025337.nc
  ✓ Opened. Variables: ['precipitation']
  Dimensions: {'time': 1, 'lat': 1800, 'lon': 3600}


...

  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251203_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251203_Global.png
Lead 3 (2025-12-04): /pesq/dados/monan/users/andre.lyra/MSWEP/MSWEP_V315_202512/daily//2025338.nc
  ✓ Opened. Variables: ['precipitation']
  Dimensions: {'time': 1, 'lat': 1800, 'lon': 3600}


/tmp/ipykernel_2546069/3220491250.py:23: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds_mswep.dims)}")


  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251204_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251204_Global.png
Lead 4 (2025-12-05): /pesq/dados/monan/users/andre.lyra/MSWEP/MSWEP_V315_202512/daily//2025339.nc
  ✓ Opened. Variables: ['precipitation']
  Dimensions: {'time': 1, 'lat': 1800, 'lon': 3600}


/tmp/ipykernel_2546069/3220491250.py:23: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(ds_mswep.dims)}")


  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251205_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MSWEP_rain_total_20251205_Global.png


## Opening MONAN data and extracting {var}

In [9]:
for exp in exps_data.keys():
    exp_name = exp
    print(f"Processing experiment: {exp_name}")
    monan_dir = exps_data[exp]["dir"]

    for var in ["RAIN", "RAINC"]:
        monan_file = f'{monan_dir}/{init}_hourly_{var_dict[var]["monan_name"]}.nc'
        monan_path = Path(monan_file)

        if os.path.exists(monan_path):
            ds = xr.open_dataset(monan_path, engine="netcdf4")
            ds = ds.assign_coords(day=ds['Time'].dt.floor('D'))
            ds_24h = ds[var_dict[var]['monan_name']].groupby('day').sum(dim='Time')
            exps_data[exp][f"ds_24h_{var_dict[var]['monan_name']}"] = ds_24h
            print(f"  ✓ Processed {var}: {var_dict[var]['monan_name']}")
        else:
            print(f"  ✗ File does not exist: {monan_path!s}")

    rain_components = []
    if "ds_24h_rainc" in exps_data[exp]:
        rain_components.append(exps_data[exp]["ds_24h_rainc"])
    if "ds_24h_rainnc" in exps_data[exp]:
        rain_components.append(exps_data[exp]["ds_24h_rainnc"])

    if rain_components:
        rain_total = rain_components[0]
        for comp in rain_components[1:]:
            rain_total = rain_total + comp
        exps_data[exp]["ds_24h_rain_total"] = rain_total

        missing = []
        if "ds_24h_rainc" not in exps_data[exp]:
            missing.append("RAINC")
        if "ds_24h_rainnc" not in exps_data[exp]:
            missing.append("RAIN")

        if missing:
            print(f"  ⚠ Calculated total rain with partial data (missing: {', '.join(missing)})")
        else:
            print("  ✓ Calculated total rain")
    else:
        print("  ✗ Could not calculate total rain (missing both RAINC and RAIN)")

print('\nProcessing complete!')

Processing experiment: teste_10deep
  ✓ Processed RAIN: rainnc
  ✓ Processed RAINC: rainc
  ✓ Calculated total rain
Processing experiment: teste_2deep
  ✓ Processed RAIN: rainnc
  ✓ Processed RAINC: rainc
  ✓ Calculated total rain
Processing experiment: teste_2deep_07liq
  ✓ Processed RAIN: rainnc
  ✓ Processed RAINC: rainc
  ✓ Calculated total rain
Processing experiment: teste_2shallow
  ✓ Processed RAIN: rainnc
  ✓ Processed RAINC: rainc
  ✓ Calculated total rain
Processing experiment: CTRL
  ✓ Processed RAIN: rainnc
  ✓ Processed RAINC: rainc
  ✓ Calculated total rain

Processing complete!


### Plotting for {UTC_hour_plot}

In [10]:
# plot for exp_name

ds_key = "ds_24h_rain_total"
plot_var_key = "RAIN"

for exp in exps_data.keys():
    exp_name = exp
    print(f"Plotting for experiment: {exp_name}")

    if ds_key not in exps_data[exp]:
        print(f"  ✗ Skipping {exp_name}: missing {ds_key}")
        continue

    monan_lons = exps_data[exp][ds_key]['lon']
    monan_lats = exps_data[exp][ds_key]['lat']
    tri = Triangulation(monan_lons, monan_lats)

    n_days = exps_data[exp][ds_key].sizes.get('day', exps_data[exp][ds_key].shape[0])

    for myext in extents.keys():
        extent_label = extents[myext]["label"]

        for lead in range(min(5, n_days)):
            fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

            date = pd.to_datetime(init, format='%Y%m%d%H') + pd.Timedelta(hours=lead*24)
            data_to_plot = exps_data[exp][ds_key][lead].values
            data_masked = np.ma.masked_where(data_to_plot < var_dict[plot_var_key]['vmin'], data_to_plot)

            tpc = ax.tripcolor(
                tri,
                data_masked,
                cmap=cmap,
                norm=norm,
                shading="flat"
            )
            ax.set_extent(extents[myext]["extent"], crs=ccrs.PlateCarree())
            plt.colorbar(
                tpc,
                label=f"{var_dict[plot_var_key]['label']} ({var_dict[plot_var_key]['unit']})",
                shrink=0.5,
                aspect=25,
                pad=0.05,
                extend='min'
            )

            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.add_feature(cfeature.STATES, linewidth=0.2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 10}
            gl.ylabel_style = {'size': 10}

            ax.set_title(
                f"MONAN {exp_name} {date.strftime('%Y-%m-%d')} + {lead*24} h lead time, {extent_label}",
                fontsize=12,
                pad=20
            )

            plt.tight_layout()
            out_png = f"{fig_path}/MONAN_{ds_key}_{init}+{lead*24}h_{exp_name}_{myext}.png"
            plt.savefig(out_png, dpi=150, bbox_inches='tight')
            #plt.show()

            print(f"Saved plot to: {out_png}")
            plt.close()

Plotting for experiment: teste_10deep
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+0h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+24h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+48h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+72h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+96h_teste_10deep_SA.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+0h_teste_10deep_Global.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+24h_teste_10deep_Global.png
Saved plot to: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_ds_24h_rain_total_2025120100+48h_

In [11]:

# plot convective fraction: ds_24h_rainc / ds_24h_rain_total

clevs_frac = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
_n_frac = len(clevs_frac) - 1
_base_frac = plt.get_cmap("YlOrRd", _n_frac - 1)
cmap_frac = ListedColormap(
    [(1, 1, 1, 1)] + [_base_frac(i) for i in range(_n_frac - 1)]
)
norm_frac = BoundaryNorm(clevs_frac, cmap_frac.N)

frac_key_num = "ds_24h_rainc"
frac_key_den = "ds_24h_rain_total"

for exp in exps_data.keys():
    exp_name = exp
    print(f"Plotting convective fraction for: {exp_name}")

    if frac_key_num not in exps_data[exp] or frac_key_den not in exps_data[exp]:
        print(f"  ✗ Skipping {exp_name}: missing {frac_key_num!r} or {frac_key_den!r}")
        continue

    monan_lons = exps_data[exp][frac_key_den]['lon']
    monan_lats = exps_data[exp][frac_key_den]['lat']
    tri_frac = Triangulation(monan_lons, monan_lats)

    n_days = exps_data[exp][frac_key_den].sizes.get('day', exps_data[exp][frac_key_den].shape[0])

    for myext in extents.keys():
        extent_label = extents[myext]["label"]

        for lead in range(min(5, n_days)):
            fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

            date = pd.to_datetime(init, format='%Y%m%d%H') + pd.Timedelta(hours=lead * 24)

            rainc = exps_data[exp][frac_key_num][lead].values
            total = exps_data[exp][frac_key_den][lead].values

            # fraction only where total > 0 to avoid division by zero
            with np.errstate(invalid='ignore', divide='ignore'):
                frac = np.where(total > 0, rainc / total, np.nan)

            frac_masked = np.ma.masked_invalid(frac)

            tpc = ax.tripcolor(
                tri_frac,
                frac_masked,
                cmap=cmap_frac,
                norm=norm_frac,
                shading="flat"
            )
            ax.set_extent(extents[myext]["extent"], crs=ccrs.PlateCarree())
            plt.colorbar(
                tpc,
                label="Convective fraction (rainc / rain_total)",
                shrink=0.5,
                aspect=25,
                pad=0.05,
                ticks=clevs_frac
            )

            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.add_feature(cfeature.STATES, linewidth=0.2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 10}
            gl.ylabel_style = {'size': 10}

            ax.set_title(
                f"MONAN {exp_name} {init} + {lead*24} h lead time, {extent_label}\n"
                f"Convective fraction",
                fontsize=12,
                pad=20
            )

            plt.tight_layout()
            out_png = f"{fig_path}/MONAN_convective_fraction_{init}+{lead*24}h_{exp_name}_{myext}.png"
            plt.savefig(out_png, dpi=150, bbox_inches='tight')
            #plt.show()

            print(f"  Saved: {out_png}")
            plt.close()


Plotting convective fraction for: teste_10deep
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+0h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+24h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+48h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+72h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+96h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+0h_teste_10deep_Global.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+24h_teste_10deep_Global.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_convective_fraction_2025120100+48h_teste_10deep_Global.png

In [12]:

# plot grid-scale fraction: ds_24h_rainnc / ds_24h_rain_total

gs_key_num = "ds_24h_rainnc"
gs_key_den = "ds_24h_rain_total"

for exp in exps_data.keys():
    exp_name = exp
    print(f"Plotting grid-scale fraction for: {exp_name}")

    if gs_key_num not in exps_data[exp] or gs_key_den not in exps_data[exp]:
        print(f"  ✗ Skipping {exp_name}: missing {gs_key_num!r} or {gs_key_den!r}")
        continue

    monan_lons = exps_data[exp][gs_key_den]['lon']
    monan_lats = exps_data[exp][gs_key_den]['lat']
    tri_gs = Triangulation(monan_lons, monan_lats)

    n_days = exps_data[exp][gs_key_den].sizes.get('day', exps_data[exp][gs_key_den].shape[0])

    for myext in extents.keys():
        extent_label = extents[myext]["label"]

        for lead in range(min(5, n_days)):
            fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection=ccrs.PlateCarree()))

            date = pd.to_datetime(init, format='%Y%m%d%H') + pd.Timedelta(hours=lead * 24)

            rainnc = exps_data[exp][gs_key_num][lead].values
            total  = exps_data[exp][gs_key_den][lead].values

            with np.errstate(invalid='ignore', divide='ignore'):
                frac_gs = np.where(total > 0, rainnc / total, np.nan)

            frac_gs_masked = np.ma.masked_invalid(frac_gs)

            tpc = ax.tripcolor(
                tri_gs,
                frac_gs_masked,
                cmap=cmap_frac,
                norm=norm_frac,
                shading="flat"
            )
            ax.set_extent(extents[myext]["extent"], crs=ccrs.PlateCarree())
            plt.colorbar(
                tpc,
                label="Grid-scale fraction (rainnc / rain_total)",
                shrink=0.5,
                aspect=25,
                pad=0.05,
                ticks=clevs_frac
            )

            ax.coastlines(linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3)
            ax.add_feature(cfeature.STATES, linewidth=0.2)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 10}
            gl.ylabel_style = {'size': 10}

            ax.set_title(
                f"MONAN {exp_name} {init} + {lead*24} h lead time, {extent_label}\n"
                f"Grid-scale fraction (rainnc / rain_total)",
                fontsize=12,
                pad=20
            )

            plt.tight_layout()
            out_png = f"{fig_path}/MONAN_rainnc_fraction_{init}+{lead*24}h_{exp_name}_{myext}.png"
            plt.savefig(out_png, dpi=150, bbox_inches='tight')
            #plt.show()

            print(f"  Saved: {out_png}")
            plt.close()


Plotting grid-scale fraction for: teste_10deep
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+0h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+24h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+48h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+72h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+96h_teste_10deep_SA.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+0h_teste_10deep_Global.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+24h_teste_10deep_Global.png
  Saved: /p/projetos/monan_atm/madeleine.gacita/figuras//MONAN_rainnc_fraction_2025120100+48h_teste_10deep_Global.png
  Saved: /p/projetos/monan_atm/

In [13]:
# Close opened xarray datasets

closed = []
failed = []
seen = set()

def try_close(obj, label):
    obj_id = id(obj)
    if obj_id in seen:
        return
    seen.add(obj_id)
    if hasattr(obj, "close") and callable(getattr(obj, "close")):
        try:
            obj.close()
            closed.append(label)
        except Exception as exc:
            failed.append((label, str(exc)))

# Close direct globals
for name, obj in list(globals().items()):
    if isinstance(obj, (xr.Dataset, xr.DataArray)):
        try_close(obj, name)

# Close xarray objects stored inside dictionaries (e.g., exps_data)
for name, obj in list(globals().items()):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, (xr.Dataset, xr.DataArray)):
                try_close(value, f"{name}[{key!r}]")
            elif isinstance(value, dict):
                for subkey, subvalue in value.items():
                    if isinstance(subvalue, (xr.Dataset, xr.DataArray)):
                        try_close(subvalue, f"{name}[{key!r}][{subkey!r}]")

print(f"Closed objects: {len(closed)}")
for item in closed:
    print(f"  - {item}")

if failed:
    print(f"\nFailed to close: {len(failed)}")
    for label, err in failed:
        print(f"  - {label}: {err}")

Closed objects: 19
  - ds_mswep
  - ds
  - ds_24h
  - rain_total
  - comp
  - monan_lons
  - monan_lats
  - exps_data['teste_10deep']['ds_24h_rainnc']
  - exps_data['teste_10deep']['ds_24h_rainc']
  - exps_data['teste_10deep']['ds_24h_rain_total']
  - exps_data['teste_2deep']['ds_24h_rainnc']
  - exps_data['teste_2deep']['ds_24h_rainc']
  - exps_data['teste_2deep']['ds_24h_rain_total']
  - exps_data['teste_2deep_07liq']['ds_24h_rainnc']
  - exps_data['teste_2deep_07liq']['ds_24h_rainc']
  - exps_data['teste_2deep_07liq']['ds_24h_rain_total']
  - exps_data['teste_2shallow']['ds_24h_rainnc']
  - exps_data['teste_2shallow']['ds_24h_rainc']
  - exps_data['teste_2shallow']['ds_24h_rain_total']
